# 05 · Mini-batches: slicing, generators, iterators, `Dataset` / `DataLoader`

The homework said: *"create three iterable objects … at each step the iterators must return two tensors"* and explicitly allowed either a custom generator/iterator or PyTorch's `Dataset` + `DataLoader`. The coding test may ask for either. You should be able to write **all** of them without help, and — more importantly — know exactly which **shapes and dtypes** come out.

---
## Theory in 60 seconds

- **Epoch** = one pass over the whole training set. **Mini-batch** = a small group of samples used for one parameter update.
- Why batches: one gradient from the full dataset is precise but expensive and gives one update per epoch; batches give many cheaper, noisier updates per epoch — and the noise even helps generalisation.
- Training batches should be drawn in a **new random order every epoch**. Evaluation order does not influence the metrics, so the test loader is not shuffled (and the homework required it to keep the order).
- An **iterable** has `__iter__()` that returns an **iterator**; an iterator has `__next__()` that returns the next item or raises `StopIteration`. A `for` loop calls `iter()` once and then `next()` repeatedly.
- A **generator function** (contains `yield`) is the short way to write an iterator. **A generator object can be consumed only once.**
- `Dataset`: knows `__len__()` and `__getitem__(idx)` for **one sample**. `DataLoader`: asks the dataset for `batch_size` samples and **stacks** them along a new first axis.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, load_houses

d = load_houses()                      # the result of notebook 04, ready-made
X_train, y_train = d["X_train"], d["y_train"]          # NumPy float32: (720, 13) and (720,)
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
print(X_train.shape, y_train.shape, X_train.dtype)

(720, 13) (720,) float32


---
## Exercise 5.1 · Batch arithmetic

720 training samples, batch size 32. Predict (no code): how many batches per epoch, and how large is the last one? Then write the general formulas for any `n` and `batch_size`.

In [2]:
n_batches_720_32 = 23             # 22 full batches (704 samples) + 1 partial
last_batch_size_720_32 = 16       # 720 - 704

def n_batches(n, batch_size):
    return (n + batch_size - 1) // batch_size        # integer ceil(n / batch_size);  math.ceil(n / batch_size) also fine

def last_batch_size(n, batch_size):
    r = n % batch_size
    return r if r != 0 else batch_size

In [3]:
check("23 batches", lambda: n_batches_720_32 == 23)
check("last batch has 16 samples", lambda: last_batch_size_720_32 == 16)
check("n_batches general", lambda: [n_batches(n, b) for n, b in ((720, 32), (64, 32), (65, 32), (5, 32))] == [23, 2, 3, 1])
check("last_batch_size general", lambda: [last_batch_size(n, b) for n, b in ((720, 32), (64, 32), (65, 32), (5, 32))] == [16, 32, 1, 5])

✅ 23 batches
✅ last batch has 16 samples
✅ n_batches general
✅ last_batch_size general


True

---
## Exercise 5.2 · Option 1: slicing inside a loop

Convert the training arrays to tensors (`Xt`: `(720, 13)`, `yt`: `(720,)`, both `float32`). Then loop over the data in consecutive batches of 32 and append the **shape pair** `(tuple(xb.shape), tuple(yb.shape))` of each batch to `shapes`. The label batch must have shape `(batch, 1)`.

Does slicing past the end (`Xt[704:736]`) raise an error?

In [4]:
Xt = torch.tensor(X_train, dtype=torch.float32)
yt = torch.tensor(y_train, dtype=torch.float32)
shapes = []
batch_size = 32
for offset in range(0, Xt.shape[0], batch_size):
    xb = Xt[offset:offset + batch_size, :]            # a slice simply stops at the end: no error, a shorter batch
    yb = yt[offset:offset + batch_size, None]         # None adds the trailing axis -> (batch, 1)
    shapes.append((tuple(xb.shape), tuple(yb.shape)))
print(len(shapes), shapes[0], shapes[-1])

23 ((32, 13), (32, 1)) ((16, 13), (16, 1))


In [5]:
check("23 batches", lambda: len(shapes) == 23)
check("full batch shapes", lambda: shapes[0] == ((32, 13), (32, 1)))
check("last batch shapes", lambda: shapes[-1] == ((16, 13), (16, 1)))

✅ 23 batches
✅ full batch shapes
✅ last batch shapes


True

---
## Exercise 5.3 · Option 2: a generator

Write `minibatch_generator(X, y, batch_size, shuffle=False, rng=None)`:

- `X`: tensor `(n, d)`, `y`: tensor `(n,)`
- yields `(xb, yb)` with `yb` of shape `(batch, 1)`
- if `shuffle` is true, the samples are visited in a random order drawn from `rng` (a `np.random.Generator`), a **new** order for every call; inputs and labels must stay paired.

<details><summary>Hint</summary>

Draw `perm = rng.permutation(n)` once at the top, then take `idx = perm[offset:offset + batch_size]` and index both tensors with `idx`. For the unshuffled case you can use `perm = np.arange(n)` and keep the same code path.
</details>

In [6]:
def minibatch_generator(X, y, batch_size, shuffle=False, rng=None):
    n = X.shape[0]
    assert y.shape == (n,)
    perm = rng.permutation(n) if shuffle else np.arange(n)
    for offset in range(0, n, batch_size):
        idx = torch.as_tensor(perm[offset:offset + batch_size])     # an index array; the same one for X and y keeps the pairs
        yield X[idx], y[idx, None]

In [7]:
def _collect(**kw):
    xs, ys = zip(*minibatch_generator(Xt, yt, 32, **kw))
    return xs, ys
check("23 batches, shapes ok", lambda: len(_collect()[0]) == 23 and _collect()[0][0].shape == (32, 13) and _collect()[1][0].shape == (32, 1) and _collect()[1][-1].shape == (16, 1))
check("unshuffled: original order", lambda: torch.equal(torch.cat(_collect()[1])[:, 0], yt))
_rng = np.random.default_rng(0)
_xs, _ys = _collect(shuffle=True, rng=_rng)
check("shuffled: a different order ...", lambda: not torch.equal(torch.cat(_ys)[:, 0], yt))
check("... but every sample exactly once", lambda: torch.equal(torch.sort(torch.cat(_ys)[:, 0]).values, torch.sort(yt).values))
check("... and inputs still paired with labels", lambda: (lambda X_, y_: all(any(torch.equal(Xt[j], X_[i]) for j in (yt == y_[i]).nonzero()[:, 0]) for i in range(40)))(torch.cat(_xs), torch.cat(_ys)[:, 0]))
_ys2 = _collect(shuffle=True, rng=_rng)[1]
check("second epoch: yet another order", lambda: not torch.equal(torch.cat(_ys2), torch.cat(_ys)))

✅ 23 batches, shapes ok
✅ unshuffled: original order
✅ shuffled: a different order ...
✅ ... but every sample exactly once
✅ ... and inputs still paired with labels
✅ second epoch: yet another order


True

---
## Exercise 5.4 · The one-epoch trap

Predict the output of this cell before running it.

In [8]:
gen = minibatch_generator(Xt, yt, 32)        # created ONCE, outside the epoch loop
for epoch in range(3):
    n_batches_seen = 0
    for xb, yb in gen:
        n_batches_seen += 1
    print(f"epoch {epoch}: {n_batches_seen} batches")

epoch 0: 23 batches
epoch 1: 0 batches
epoch 2: 0 batches


What happened, what would the loss curve of such a training look like, and how do you fix it?

**Answer.** A generator object is an *iterator*: once it has run to its end it is exhausted and every further `for` loop over it ends immediately. Only epoch 0 trains; epochs 1, 2, … do **zero** updates. The symptom: the loss improves in the first epoch and is then perfectly flat (and if you average an empty list of batch losses you get `nan` or a warning). Fix: create a **new** generator in every epoch — `for xb, yb in minibatch_generator(Xt, yt, 32):` inside the epoch loop — or wrap it in an *iterable* object whose `__iter__` creates a fresh iterator each time (next exercise). A `DataLoader` is such an iterable, which is why you can create it once.

---
## Exercise 5.5 · Option 3: iterator and iterable classes

**a)** Write the explicit iterator `MinibatchIterator` (state: the current offset; `__next__` returns the next batch or raises `StopIteration`). No shuffling needed.

**b)** Write the iterable `BatchLoader`, which can be looped over **many times**: `__iter__` returns a fresh iterator each time (shuffled if requested), `__len__` returns the number of batches.

In [9]:
class MinibatchIterator:
    def __init__(self, X, y, batch_size):
        self.X, self.y, self.batch_size = X, y, batch_size
        self.offset = 0                               # the "bookmark"

    def __iter__(self):
        return self                                   # an iterator is its own iterable

    def __next__(self):
        if self.offset >= self.X.shape[0]:
            raise StopIteration()                     # tells the for loop to stop
        xb = self.X[self.offset:self.offset + self.batch_size]
        yb = self.y[self.offset:self.offset + self.batch_size, None]
        self.offset += self.batch_size
        return xb, yb


class BatchLoader:
    def __init__(self, X, y, batch_size, shuffle=False, seed=0):
        self.X, self.y, self.batch_size, self.shuffle = X, y, batch_size, shuffle
        self.rng = np.random.default_rng(seed)        # lives across epochs -> a new order each epoch

    def __len__(self):
        return (self.X.shape[0] + self.batch_size - 1) // self.batch_size

    def __iter__(self):                               # called at the start of every for loop
        return minibatch_generator(self.X, self.y, self.batch_size, self.shuffle, self.rng)

In [10]:
check("MinibatchIterator: 23 batches with the right shapes", lambda: [yb.shape[0] for _, yb in MinibatchIterator(Xt, yt, 32)] == [32] * 22 + [16])
check("MinibatchIterator: next() works step by step", lambda: (lambda it: next(it)[0].shape == (100, 13) and torch.equal(next(it)[1][:, 0], yt[100:200]))(MinibatchIterator(Xt, yt, 100)))
check("MinibatchIterator is exhausted after one pass", lambda: (lambda it: (len(list(it)), len(list(it))))(MinibatchIterator(Xt, yt, 32)) == (23, 0))
_bl = None
try: _bl = BatchLoader(Xt, yt, 32, shuffle=True)
except NotImplementedError: pass
check("BatchLoader: len", lambda: len(_bl) == 23)
check("BatchLoader: can be iterated in every epoch", lambda: [len(list(_bl)) for _ in range(3)] == [23, 23, 23])
check("BatchLoader: a new order each epoch", lambda: not torch.equal(torch.cat([y for _, y in _bl]), torch.cat([y for _, y in _bl])))

✅ MinibatchIterator: 23 batches with the right shapes
✅ MinibatchIterator: next() works step by step
✅ MinibatchIterator is exhausted after one pass
✅ BatchLoader: len
✅ BatchLoader: can be iterated in every epoch
✅ BatchLoader: a new order each epoch


True

---
## Exercise 5.6 · Option 4: `Dataset` + `DataLoader`

Write `RegressionDataset`: the constructor receives NumPy arrays `x` `(n, d)` and `y` `(n,)` and stores them as `float32` tensors; `__getitem__(idx)` returns the pair for **one** sample such that a `DataLoader` yields batches of shape `(batch, d)` and `(batch, 1)`.

Then create `loader_train`, `loader_val` (both shuffled) and `loader_test` (not shuffled), batch size 32.

In [11]:
class RegressionDataset(Dataset):
    def __init__(self, x, y):
        self.xs = torch.tensor(x, dtype=torch.float32)       # (n, d)
        self.ys = torch.tensor(y, dtype=torch.float32)       # (n,)

    def __len__(self):
        return self.xs.shape[0]

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx, None]              # (d,) and (1,)  -> stacked to (batch, d) and (batch, 1)

loader_train = DataLoader(RegressionDataset(X_train, y_train), batch_size=32, shuffle=True)
loader_val = DataLoader(RegressionDataset(X_val, y_val), batch_size=32, shuffle=True)
loader_test = DataLoader(RegressionDataset(X_test, y_test), batch_size=32, shuffle=False)

In [12]:
_ds = None
try: _ds = RegressionDataset(X_train, y_train)
except NotImplementedError: pass
check("len(dataset) == 720", lambda: len(_ds) == 720)
check("one sample: shapes (13,) and (1,), float32", lambda: _ds[0][0].shape == (13,) and _ds[0][1].shape == (1,) and _ds[0][0].dtype == _ds[0][1].dtype == torch.float32)
check("train batch shapes (32, 13) and (32, 1)", lambda: (lambda b: b[0].shape == (32, 13) and b[1].shape == (32, 1))(next(iter(loader_train))))
check("len(loader_train) == 23 batches", lambda: len(loader_train) == 23)
check("train loader shuffles", lambda: not torch.equal(torch.cat([y for _, y in loader_train])[:, 0], torch.tensor(y_train)))
check("test loader keeps the order", lambda: torch.equal(torch.cat([y for _, y in loader_test])[:, 0], torch.tensor(y_test)))

✅ len(dataset) == 720
✅ one sample: shapes (13,) and (1,), float32
✅ train batch shapes (32, 13) and (32, 1)
✅ len(loader_train) == 23 batches
✅ train loader shuffles
✅ test loader keeps the order


True

---
## Exercise 5.7 · What does the `DataLoader` stack? Predict the batch shapes

The `DataLoader` takes `batch_size` results of `__getitem__` and stacks each element along a **new first axis**. So the label shape of a batch is decided entirely by what `__getitem__` returns for one sample.

For each variant predict the shape of the label batch (batch size 32) and say which loss it suits.

In [13]:
class Variant(Dataset):
    """ys_shape: how the labels are stored; index: how __getitem__ indexes them."""
    def __init__(self, ys, index):
        self.ys, self.index = ys, index
    def __len__(self):
        return self.ys.shape[0]
    def __getitem__(self, idx):
        return self.index(self.ys, idx)

ys_1d = torch.tensor(y_train)                  # (720,)
ys_2d = torch.tensor(y_train)[:, None]         # (720, 1)
variants = {
    "A: ys (n,)   -> ys[idx]":        Variant(ys_1d, lambda ys, i: ys[i]),
    "B: ys (n,)   -> ys[idx, None]":  Variant(ys_1d, lambda ys, i: ys[i, None]),
    "C: ys (n, 1) -> ys[idx]":        Variant(ys_2d, lambda ys, i: ys[i]),
    "D: ys (n, 1) -> ys[idx, None]":  Variant(ys_2d, lambda ys, i: ys[i, None]),
}

In [14]:
pred_shapes = {
    "A": (32,),          # each sample is a scalar tensor ()      -> stacked: (32,)       what CrossEntropyLoss wants (with int64)
    "B": (32, 1),        # each sample is (1,)                    -> stacked: (32, 1)     MSELoss / BCELoss with a (batch, 1) output
    "C": (32, 1),        # each sample is the row (1,)            -> stacked: (32, 1)     same
    "D": (32, 1, 1),     # each sample is (1, 1)                  -> stacked: (32, 1, 1)  one axis too many -> broadcasting bug with MSE
}

In [15]:
for name, ds in variants.items():
    actual = tuple(next(iter(DataLoader(ds, batch_size=32))).shape)
    check(f"{name:32s} -> {actual}", lambda: tuple(pred_shapes[name[0]]) == actual)

✅ A: ys (n,)   -> ys[idx]          -> (32,)
✅ B: ys (n,)   -> ys[idx, None]    -> (32, 1)
✅ C: ys (n, 1) -> ys[idx]          -> (32, 1)
✅ D: ys (n, 1) -> ys[idx, None]    -> (32, 1, 1)


---
## Exercise 5.8 · The average loss of an epoch — exactly

The lecturer's notebook notes that averaging the per-batch losses is *"not exactly precise because of the last (possibly) incomplete batch"*. Make that concrete.

For the fixed (untrained) model below and `loader_test` (240 samples, **batch size 50** here, so the last batch has 40), compute:

- `mean_of_batch_means`: the plain average of the per-batch MSE values,
- `weighted_mean`: each batch loss weighted by its number of samples, divided by the total number of samples,
- `full_loss`: the MSE computed on the whole test set in one go.

Which two agree, and why?

In [16]:
torch.manual_seed(0)
model = nn.Linear(13, 1)
loss_fn = nn.MSELoss()
loader50 = DataLoader(RegressionDataset(X_test, y_test), batch_size=50, shuffle=False)

batch_losses, batch_sizes = [], []
with torch.no_grad():                                   # evaluation: no graph needed
    for xb, yb in loader50:
        batch_losses.append(loss_fn(model(xb), yb).item())
        batch_sizes.append(xb.shape[0])
    full_loss = loss_fn(model(torch.tensor(X_test)), torch.tensor(y_test)[:, None]).item()

mean_of_batch_means = float(np.mean(batch_losses))
weighted_mean = float(np.sum(np.array(batch_losses) * np.array(batch_sizes)) / np.sum(batch_sizes))
print(mean_of_batch_means, weighted_mean, full_loss)

49263.171875 49287.596028645836 49287.59765625


In [17]:
check("weighted mean == loss on the full set", lambda: abs(weighted_mean - full_loss) / full_loss < 1e-5)
check("plain mean of batch means is (slightly) different", lambda: abs(mean_of_batch_means - full_loss) / full_loss > 1e-5 and abs(mean_of_batch_means - full_loss) / full_loss < 0.2)

✅ weighted mean == loss on the full set
✅ plain mean of batch means is (slightly) different


True

**Answer.** The weighted mean equals the full-set loss: MSE is a mean over samples, so the total is `Σ_batches (batch mean × batch size) / n`. The plain mean of batch means gives the small last batch the same weight as a full one, so its samples count more than the others. For **training curves** the plain mean is fine (the homework solution and the lecturer use it). For a **reported test metric** be exact: either weight by batch size, or — simplest, and what the homework did — concatenate all predictions and labels with `torch.cat` and compute the loss/metric once on the full tensors.

---
## Exercise 5.9 · A classification dataset

For multi-class classification the labels must come out of the loader as shape `(batch,)` and dtype `int64`. Write `ClassificationDataset` accordingly and build a shuffled loader with batch size 16 for the toy data below.

In [18]:
rng = np.random.default_rng(1)
X_toy = rng.normal(size=(100, 4)).astype(np.float32)
y_toy = rng.integers(0, 3, size=100)                 # NumPy int64, values 0/1/2

class ClassificationDataset(Dataset):
    def __init__(self, x, y):
        self.xs = torch.tensor(x, dtype=torch.float32)
        self.ys = torch.tensor(y, dtype=torch.int64)         # class indices

    def __len__(self):
        return self.xs.shape[0]

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]                    # scalar label -> batches of shape (batch,)

loader_toy = DataLoader(ClassificationDataset(X_toy, y_toy), batch_size=16, shuffle=True)

In [19]:
check("batch shapes (16, 4) and (16,)", lambda: (lambda b: b[0].shape == (16, 4) and b[1].shape == (16,))(next(iter(loader_toy))))
check("dtypes float32 and int64", lambda: (lambda b: b[0].dtype == torch.float32 and b[1].dtype == torch.int64)(next(iter(loader_toy))))
check("7 batches, last one has 4 samples", lambda: [len(y) for _, y in loader_toy] == [16] * 6 + [4])
check("usable with CrossEntropyLoss", lambda: nn.CrossEntropyLoss()(nn.Linear(4, 3)(next(iter(loader_toy))[0]), next(iter(loader_toy))[1]).ndim == 0)

✅ batch shapes (16, 4) and (16,)
✅ dtypes float32 and int64
✅ 7 batches, last one has 4 samples
✅ usable with CrossEntropyLoss


True

---
## Blank-page drill

Cover the notebook. In the empty cell write from memory, for the arrays `X_train, y_train, X_val, y_val, X_test, y_test` (regression):

1. a `Dataset` subclass,
2. three `DataLoader`s with the correct `shuffle` settings,
3. a sanity check that prints the shape and dtype of one batch,
4. **and** the generator version `minibatch_generator` (with shuffling).

Self-check: `(32, 13) float32` and `(32, 1) float32`.

In [20]:
class MyDataset(Dataset):
    def __init__(self, x, y):
        self.xs = torch.tensor(x, dtype=torch.float32)
        self.ys = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return self.xs.shape[0]
    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx, None]

dl_train = DataLoader(MyDataset(X_train, y_train), batch_size=32, shuffle=True)
dl_val = DataLoader(MyDataset(X_val, y_val), batch_size=32, shuffle=True)
dl_test = DataLoader(MyDataset(X_test, y_test), batch_size=32, shuffle=False)
xb, yb = next(iter(dl_train))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

def my_generator(X, y, batch_size, rng):
    perm = rng.permutation(X.shape[0])
    for offset in range(0, X.shape[0], batch_size):
        idx = torch.as_tensor(perm[offset:offset + batch_size])
        yield X[idx], y[idx, None]

xb, yb = next(my_generator(torch.tensor(X_train), torch.tensor(y_train), 32, np.random.default_rng(0)))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

torch.Size([32, 13]) torch.float32 torch.Size([32, 1]) torch.float32
torch.Size([32, 13]) torch.float32 torch.Size([32, 1]) torch.float32
